# Vibronic coupling in tetracene dimers

## General post-processing / prototyping notebook

### Can be done locally:
- Boys localisation and partial diagonalisation (use populations from checkfile)
- orbital visualisation
- transform EPH coupling to localised subspace

In [15]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import os
import webbrowser
import py3Dmol
from pyscf import lib
from pyscf.tools import cubegen
from pyscfutils import xyz_string, mf_from_chk, Boys_frontier_orbitals, \
                       pair_neighboring_orbitals, block_frontier_orbitals, \
                       fock_matrix

In [2]:
# directories for source data and results
datadir = os.path.realpath('../runs/10264877/')
savedir = os.path.realpath('../results/')

# load checkfile and use it to create mol and mf object
mf = mf_from_chk(datadir + '/' + 'calc.chk')
mol = mf.mol

Calculate Fock matrix in the atomic orbital basis (time consuming).

In [4]:
fock_ao = fock_matrix(mf)

Localize the 4 frontier molecular orbitals using Boys localisation. We expect two Boys orbitals on each tetracene fragment. Reorder the molecular orbitals orbitals so that they appear one pair after another. After that, we block-diagonalize the Fock matrix to obtain localized orthogonal orbitals corresponding to HOMO and LUMO of each tetracene unit.

In [8]:
mo_coeff_boys, _ = Boys_frontier_orbitals(mf, 4)
mo_coeff_boys, idx = pair_neighboring_orbitals(mf, mo_coeff_boys)
mo_coeff_block = block_frontier_orbitals(mf, mo_coeff_boys, fock=fock_ao)

Save orbitals as html files. These can be rendered by py3Dmol inside a Jupyter notebook, or opened in an external browser. Visualization in an external browser helps keep the notebook nice and lightweight.

In [14]:
for i, mo in enumerate(mo_coeff_boys.T):
    orbname = savedir + '/' + 'test_orbBoys' + str(i+1) + '.cube'
    cubegen.orbital(mol, orbname, mo, nx=100, ny=50, nz=50)
    v = view_orbital(orbname, mol, iso=0.02, alpha=0.9, html=True)

for i, mo in enumerate(mo_coeff_block.T):
    orbname = savedir + '/' + 'test_orbBlock' + str(i+1) + '.cube'
    cubegen.orbital(mol, orbname, mo, nx=100, ny=50, nz=50)
    v = view_orbital(orbname, mol, iso=0.02, alpha=0.9, html=True)


In [4]:
"""
Order Boys orbitals according to the molecular fragment on which they are localized:
left front (idx1), left back (idx2), right front (idx3), right back (idx4)
"""
mo_front_Boys, _ = Boys_frontier_orbitals(mf)
pop_front_Boys = atomic_populations(mf, mo_front_Boys)
fragment_atom_idx = np.array([idx1, idx2, idx3, idx4], dtype=int)
fragment_Boys_overlap = fragment_atom_idx @ pop_front_Boys
idx1234 = np.argmax(fragment_Boys_overlap, axis=1)

mo_front_Boys = mo_front_Boys.copy()[:,idx1234]

#mo_Boys = mo_coeff.copy()
#mo_Boys[:,frontier_idx] = mo_front_Boys

In [ ]:
# --- Export frontier orbitals as cube files ---
for i in range(mo_front_Boys.shape[1]):
    cubegen.orbital(mol, 
                    savedir + '/' + 'orbBoys' + str(i+1) + '.cube', 
                    mo_front_Boys[:,i],
                    nx=400, ny=200, nz=200)

### Visualise localised orbitals

In [ ]:
def view_orbital(cube_filename, mol_obj, iso=0.025, alpha=0.9, html=True):
    """
    Visualize orbital from cube file. Generate view with py3Dmol which can be displayed
    as view.show() in a Jupyter notebook. Optionally save orbital as html and visualize
    it in an external browser. This helps keep the notebook nice and lightweight.
    """
    with open(cube_filename) as f:
        cube_data = f.read()
    view = py3Dmol.view(width=600, height=300)
    symbols = mol_obj.elements
    coords = mol_obj.atom_coords(unit="Angstrom")
    view.addModel(xyz_string(symbols, coords), "xyz")
    view.setStyle({"model":0}, {"stick":{"colorscheme":"grayCarbon"}})
    view.addVolumetricData(cube_data, "cube", {
        "isoval": iso,
        "color": "blue",
        "opacity": alpha
    })
    view.addVolumetricData(cube_data, "cube", {
        "isoval": -iso,
        "color": "red",
        "opacity": alpha
    })
    view.setViewStyle({"style": "outline"})
    view.setBackgroundColor("white")
    view.zoomTo()
    view.rotate(-45, "x")
    view.zoom(1.5)
    if html:
        htmlview = view._make_html()
        html_filename = cube_filename.split(".")[0] + ".html"
        with open(html_filename, "w") as f:
            f.write(htmlview)
        webbrowser.open("file://" + os.path.realpath(html_filename))
    return view

### To do: 
### check fock matri

In [ ]:
# --- Fock matrix in the localised basis ---
dm = mf.make_rdm1()
Fao = mf.get_fock(dm=dm)
FBoys = mo_front_Boys.T @ Fao @ mo_front_Boys
print(FBoys)

In [ ]:
# --- Block diagonalisation ---
FLeft = FBoys[:2,:2]
FRight = FBoys[-2:,-2:]
def eigsort(M):
    val, vec = np.linalg.eig(M)
    i = np.argsort(val)
    return val[i], vec[:,i]
valLeft,  vecLeft = eigsort(FLeft)
valRight, vecRight = eigsort(FRight)
UBoys2loc = np.block([[vecLeft,np.zeros((2,2))],[np.zeros((2,2)), vecRight]])
mo_front_block = mo_front_Boys @ UBoys2loc
print(UBoys2loc.T @ FBoys @ UBoys2loc)

In [ ]:
#print('Fock matrix in the MO basis (invcm)')
#print(mo_front.T @ Fao @ mo_front * 219474.63)
#print('')
#print('Fock matrix in the Boys basis (invcm)')
#print(FBoys * 219474.63)
#print('')
#print('Fock matrix in the block-diagonal basis (invcm)')
#print(UBoys2loc.T @ FBoys @ UBoys2loc * 219474.63)

In [ ]:
# --- Export block-localised orbitals as cube files ---
for i, orb_idx in enumerate(frontier_idx):
    cubegen.orbital(mol, 
                    savedir + '/' + 'orbBlock' + str(i+1) + '.cube', 
                    mo_front_block[:, i],
                    nx=400, ny=200, nz=200)

In [ ]:
for orbName in ['orbBlock1.cube', 'orbBlock2.cube', 'orbBlock3.cube', 'orbBlock4.cube']:
    v = view_orbital(savedir + '/' + orbName, 
                     mol, 
                     iso=0.02, 
                     alpha=0.9, 
                     html=True)

### Extract two-electron integrals within the frontier subspace
Note: PySCF uses chemists notation, i.e.
$$
( i j | k l ) = \langle i k | j l \rangle = 
\int dx_1 \int dx_2 \chi^*_i(x_1) \chi^*_k(x_2) r^{-1}_{12} \chi_j(x_1) \chi_l(x_2) 
$$

and the frontier block-diagonalised orbitals are stored in the order:

left homo, left lumo, right homo, right lumo

In [ ]:
from pyscf import ao2mo
eri4mo = ao2mo.kernel(mol, mo_front_block)
eri4mo = ao2mo.restore(1, eri4mo, 4)

According to Smith & Michl, Chem. Rev. 110, 6891-6936 (2010), the CI Hamiltonian matrix elements within the subspace spanned by singlet excitons (SE), charge transfer states (CT), and singlet triplet-triplet (TT) are
$$
\begin{align}
\langle \mathrm{CT}_A|H|\mathrm{CT}_B\rangle &= 2(h_Al_B|l_Ah_B)-(h_Ah_B|l_Al_B) \\
\langle \mathrm{SE}_A|H|\mathrm{SE}_B\rangle &= 2(h_Al_A|l_Bh_B)-(h_Ah_B|l_Bl_A) \\
\langle \mathrm{CT}_A|H|\mathrm{SE}_A\rangle &= \langle l_A|F|l_B \rangle+2(h_Al_B|l_Ah_A)-(h_Ah_A|l_Al_B) \\
\langle \mathrm{CT}_A|H|\mathrm{SE}_B\rangle &= \langle h_A|F|h_B \rangle+2(h_Bl_B|l_Bh_A)-(h_Bh_A|l_Bl_B) \\
\langle \mathrm{SE}_A|H|\mathrm{TT}\rangle &= \sqrt{\frac{3}{2}} \left[(l_Ah_B|l_Bl_A)-(h_Al_B|h_Bh_A)\right]\\
\langle \mathrm{CT}_A|H|\mathrm{TT}\rangle &= \sqrt{\frac{3}{2}} \left[ \langle l_A|F|h_B\rangle + (l_Ah_B|l_Bl_B)-(l_Ah_B|h_Ah_A) \right]
\end{align}
$$
(all other matrix elements can be obtained by switching $A\leftrightarrow B$) and using hermiticity)

In [ ]:
# tewo-electron integrals needed for TT exchange coupliong (Kollmar 1993)
JhlL = eri4mo[0,0,1,1]
JhlR = eri4mo[2,2,3,3]
KhlL = eri4mo[0,1,1,0]
KhlR = eri4mo[2,3,3,2]

print(JhlL, JhlR)
print(KhlL, KhlR)


## Import vibronic coupling

In [ ]:
# --- Read electron-phonon matrix (convert to invcm) ---
#h5file = datadir + '/' + 'out.h5'
h5file = datadir + '/' + 'dft-lvc-out.h5'
with h5py.File(h5file, 'r') as f:
    ephmat  = np.array(list(f['ephmat']))  * 219474.63
    omega   = np.array(list(f['omega']))   * 219474.63
    modevec = np.array(list(f['modevec']))

In [ ]:
# --- Convert ephmat to vibronic coupling in the localised subspace ---
g = np.einsum('pi, kij, jq -> kpq',
              mo_front_block.T,
              ephmat, 
              mo_front_block, 
              optimize=True)


# --- Convert modevec to Cartesian coordinates ---
modevec_cart = modevec.reshape(mol.natm, 3, modevec.shape[-1])
mass = mol.atom_mass_list()
for i in range(len(mass)):
    modevec_cart[i] = modevec_cart[i] / np.sqrt(mass[i])


# --- Vibronic coupling in the multiconfigurational basis LE+, LE-, CT+, CT-, TT ---
# index ordering hA  lA  hB  lB  (homo/lumo, left/right)
# index ordering SE+ SE- CT+ CT- TT
nvib = omega.shape[0]
W = np.zeros((nvib,5,5))
W[:,0,1] = 0.5 * ( g[:,2,2] - g[:,0,0] - g[:,3,3] + g[:,1,1] )
W[:,2,3] = 0.5 * ( g[:,2,2] - g[:,0,0] + g[:,3,3] - g[:,1,1] )
W[:,0,2] = g[:,1,3] - g[:,0,3]
W[:,1,3] = g[:,1,3] + g[:,0,3]
W[:,4,2] = np.sqrt(3) / 2 * ( g[:,1,2] + g[:,0,3] )
W[:,4,3] = np.sqrt(3) / 2 * ( g[:,1,2] - g[:,0,3] )
W = W + W.transpose(0,2,1)



In [ ]:
#modevec_mw = np.einsum(' -> ', np.sqrt(mol.atom_mass_list()), modevec)

In [ ]:
a, k = 0.5, 0

mol_eq = lib.chkfile.load_mol('calc.chk')
xyz_eq = xyz_string(mol_eq.elements, mol_eq.atom_coords() * 0.529177)
xyz_pv = xyz_string(mol_eq.elements, 
                    (mol_eq.atom_coords() + a * modevec_cart[:,:,k]) * 0.529177)
xyz_nv = xyz_string(mol_eq.elements, 
                    (mol_eq.atom_coords() - a * modevec_cart[:,:,k]) * 0.529177)

view = py3Dmol.view(width=600, height=300)
view.clear()
view.addModel(xyz_eq, 'xyz')
view.addModel(xyz_pv, 'xyz')
view.addModel(xyz_nv, 'xyz')

#view.setStyle({'model':0}, {'stick':{'colorscheme':'grayCarbon', 'radius':0.05}})
view.setStyle({'model':1}, 
              {'stick':{'color':'red', 'radius':0.02},
               'sphere':{'color':'red', 'radius':0.1}})
view.setStyle({'model':2}, 
              {'stick':{'color':'blue', 'radius':0.02},
               'sphere':{'color':'blue', 'radius':0.1}})

view.setBackgroundColor('white')
view.zoomTo()
view.zoom(4)
view.show()


In [ ]:
modevec[:,0].reshape(5,3)


In [ ]:
mol_eq.atom_coords()

## Idea:
### We already have the vibronic coupling in the full MO basis - let's use it!

Instead of doing SVD to identify the vibronic coupling patterns in a small pre-selected electroinc state manifold, we could use the information from more than those four frontier molecular orbitals?

This is the same as identifying a preferred vibronic basis?
Identify hieararchies with SVD.